# Manufacturing – Predictive Maintenance: Synthetic Event Generator

Streams synthetic equipment sensor telemetry for **4 machines** into Microsoft Fabric via Eventstream Custom Endpoint.

**Upload and run inside Microsoft Fabric.**

---

## ⚡ Demo Quick-Start
1. **Cell 1** – Install packages
2. **Cell 2** – Paste connection string
3. **Cell 3** – Load machine & scenario definitions
4. **Cell 4** – Load generator functions
5. **Cell 5** – Start background streaming
6. **Cell 6** – Demo trigger (crisis/anomaly on demand)

In [ ]:
# Cell 1 — Install dependencies
!pip install azure-eventhub faker --quiet

In [ ]:
# Cell 2 — Configuration
# Paste your Eventstream Custom Endpoint credentials here.
# Find these in Fabric: Eventstream → Custom Endpoint → Keys tab

EVENT_HUB_NAME        = "<paste-your-event-hub-name>"
EVENT_HUB_CONN_STRING = "<paste-your-connection-string>"

# Generator settings
EVENTS_PER_BATCH      = 8      # events sent per cycle
INTERVAL_SECONDS      = 3      # seconds between batches
AUTO_CRISIS_ENABLED   = True   # random crisis bursts
AUTO_MACRO_ENABLED    = True   # macro event injection

print("✅ Configuration loaded.")

In [ ]:
# Cell 3 — Machine & Scenario Definitions
import random

MACHINES = [
    {
        "machineId": "CNC-007",
        "machineName": "CNC Milling Machine #7",
        "location": "Plant A – Bay 3",
        "sensors": [
            {"type": "Temperature", "unit": "°C",    "normal_range": (35, 70),  "anomaly_range": (85, 105)},
            {"type": "Vibration",   "unit": "mm/s",  "normal_range": (0.5, 4),  "anomaly_range": (10, 18)},
            {"type": "SpindleRPM", "unit": "RPM",   "normal_range": (2000, 4500), "anomaly_range": (500, 1200)}
        ],
        "crisis_issues": ["Motor overheat", "Spindle bearing failure", "Lubrication breakdown"],
        "crisis_keywords": ["overheat", "spindle", "bearing", "seize", "thermal"]
    },
    {
        "machineId": "HYD-003",
        "machineName": "Hydraulic Press Line 3",
        "location": "Plant A – Bay 7",
        "sensors": [
            {"type": "Pressure",    "unit": "bar",   "normal_range": (8, 14),   "anomaly_range": (2, 5)},
            {"type": "Vibration",   "unit": "mm/s",  "normal_range": (0.3, 3),  "anomaly_range": (9, 16)},
            {"type": "Temperature", "unit": "°C",    "normal_range": (40, 65),  "anomaly_range": (80, 100)}
        ],
        "crisis_issues": ["Hydraulic leak", "Pump failure", "Pressure loss"],
        "crisis_keywords": ["pressure", "leak", "pump", "hydraulic", "rupture"]
    },
    {
        "machineId": "CMP-A1",
        "machineName": "Industrial Air Compressor A1",
        "location": "Plant B – Utility Room",
        "sensors": [
            {"type": "Temperature", "unit": "°C",    "normal_range": (50, 80),  "anomaly_range": (95, 120)},
            {"type": "Vibration",   "unit": "mm/s",  "normal_range": (0.5, 3.5),"anomaly_range": (8, 14)},
            {"type": "Current",     "unit": "A",     "normal_range": (30, 55),  "anomaly_range": (65, 85)}
        ],
        "crisis_issues": ["Power surge", "Motor overload", "Oil overheating"],
        "crisis_keywords": ["current", "surge", "overload", "oil", "electrical"]
    },
    {
        "machineId": "CNV-012",
        "machineName": "Conveyor Belt Unit 12",
        "location": "Plant B – Assembly Line 2",
        "sensors": [
            {"type": "Torque",      "unit": "Nm",    "normal_range": (5, 20),   "anomaly_range": (30, 50)},
            {"type": "Speed",       "unit": "m/min", "normal_range": (10, 25),  "anomaly_range": (2, 6)},
            {"type": "Temperature", "unit": "°C",    "normal_range": (30, 55),  "anomaly_range": (70, 90)}
        ],
        "crisis_issues": ["Belt slip", "Motor stall", "Recurring stops"],
        "crisis_keywords": ["torque", "slip", "stall", "jam", "conveyor"]
    }
]

OPERATIONAL_MODES = ["Production", "Idle", "Warmup", "Cooldown"]
MODE_WEIGHTS      = [0.70, 0.15, 0.10, 0.05]

MACRO_EVENTS = [
    {
        "id": "spare_parts_shortage",
        "label": "Global Spare Parts Shortage",
        "affected_machines": ["CNC-007", "HYD-003"],
        "effect": "degraded",
        "description": "Bearing and seal supply disruption — machines running beyond service intervals"
    },
    {
        "id": "energy_price_spike",
        "label": "Energy Price Spike",
        "affected_machines": ["CMP-A1", "CNV-012"],
        "effect": "stressed",
        "description": "Electricity costs surge 40% — equipment pushed to run at max efficiency"
    },
    {
        "id": "safety_regulation",
        "label": "New Safety Regulation Enforcement",
        "affected_machines": ["CNC-007", "HYD-003", "CMP-A1", "CNV-012"],
        "effect": "heightened_monitoring",
        "description": "Stricter HSE requirements mandate continuous monitoring of all critical assets"
    }
]

print(f"✅ Loaded {len(MACHINES)} machines, {len(MACRO_EVENTS)} macro events.")

In [ ]:
# Cell 4 — Generator Functions
import uuid
import json
import time
import threading
from datetime import datetime, timezone
from azure.eventhub import EventHubProducerClient, EventData

# ── State ──────────────────────────────────────────────────
GENERATOR_STATE = {
    "running": False,
    "force_crisis_now": False,
    "crisis_target_machine": None,
    "active_macro": None,
    "total_sent": 0,
    "anomaly_sent": 0,
    "crisis_triggered": 0
}

# ── Helpers ────────────────────────────────────────────────
def _pick_sensor(machine):
    return random.choice(machine["sensors"])

def _normal_reading(sensor):
    lo, hi = sensor["normal_range"]
    return round(random.uniform(lo, hi), 2)

def _anomaly_reading(sensor):
    lo, hi = sensor["anomaly_range"]
    return round(random.uniform(lo, hi), 2)

def _status_code(is_anomaly, is_crisis):
    if is_crisis:
        return 2  # Fault
    if is_anomaly:
        return 1  # Warning
    return 0      # OK

def _should_inject_anomaly():
    """15% chance of an anomaly reading under normal conditions."""
    return random.random() < 0.15

def _should_auto_crisis():
    """~2% chance each cycle to auto-trigger a short crisis burst."""
    return AUTO_CRISIS_ENABLED and random.random() < 0.02

def _should_auto_macro():
    """~1% chance each cycle to activate a macro event for 30 seconds."""
    return AUTO_MACRO_ENABLED and random.random() < 0.01

# ── Single Event Builder ──────────────────────────────────
def generate_event(machine=None, force_anomaly=False, force_crisis=False):
    if machine is None:
        machine = random.choice(MACHINES)
    
    sensor = _pick_sensor(machine)
    is_crisis = force_crisis
    is_anomaly = force_anomaly or _should_inject_anomaly()
    
    if is_crisis or is_anomaly:
        value = _anomaly_reading(sensor)
    else:
        value = _normal_reading(sensor)
    
    mode = random.choices(OPERATIONAL_MODES, weights=MODE_WEIGHTS, k=1)[0]
    if is_crisis:
        mode = "Production"  # crises happen during production
    
    event = {
        "eventId":          str(uuid.uuid4()),
        "timestamp":        datetime.now(timezone.utc).isoformat(),
        "machineId":        machine["machineId"],
        "machineName":      machine["machineName"],
        "sensorType":       sensor["type"],
        "sensorValue":      str(value),
        "unit":             sensor["unit"],
        "statusCode":       str(_status_code(is_anomaly, is_crisis)),
        "maintenanceFlag":  str(False),
        "operationalMode":  mode,
        "location":         machine["location"]
    }
    return event

# ── Batch Sender ──────────────────────────────────────────
def send_batch(events):
    producer = EventHubProducerClient.from_connection_string(
        conn_str=EVENT_HUB_CONN_STRING,
        eventhub_name=EVENT_HUB_NAME
    )
    try:
        batch = producer.create_batch()
        for e in events:
            batch.add(EventData(json.dumps(e).encode("utf-8")))
        producer.send_batch(batch)
    finally:
        producer.close()

# ── Background Loop ───────────────────────────────────────
def _generator_loop():
    crisis_burst_remaining = 0
    crisis_machine = None
    macro_expire = 0
    
    while GENERATOR_STATE["running"]:
        events = []
        now = time.time()
        
        # ── Check for forced crisis trigger ──
        if GENERATOR_STATE["force_crisis_now"]:
            crisis_burst_remaining = 20  # 20 batches of crisis data
            target = GENERATOR_STATE.get("crisis_target_machine")
            crisis_machine = next((m for m in MACHINES if m["machineId"] == target), random.choice(MACHINES))
            GENERATOR_STATE["force_crisis_now"] = False
            GENERATOR_STATE["crisis_triggered"] += 1
            print(f"🚨 CRISIS triggered on {crisis_machine['machineName']}!")
        
        # ── Auto-crisis ──
        if crisis_burst_remaining == 0 and _should_auto_crisis():
            crisis_burst_remaining = 10
            crisis_machine = random.choice(MACHINES)
            GENERATOR_STATE["crisis_triggered"] += 1
            print(f"⚠️  Auto-crisis on {crisis_machine['machineName']}")
        
        # ── Auto-macro ──
        if GENERATOR_STATE["active_macro"] is None and _should_auto_macro():
            macro = random.choice(MACRO_EVENTS)
            GENERATOR_STATE["active_macro"] = macro
            macro_expire = now + 30  # active for 30 seconds
            print(f"🌍 Macro event: {macro['label']}")
        
        if GENERATOR_STATE["active_macro"] and now > macro_expire:
            print(f"🌍 Macro event ended: {GENERATOR_STATE['active_macro']['label']}")
            GENERATOR_STATE["active_macro"] = None
        
        # ── Build event batch ──
        for _ in range(EVENTS_PER_BATCH):
            if crisis_burst_remaining > 0:
                # Mix: ~60% crisis events for the target machine, 40% normal
                if random.random() < 0.6:
                    events.append(generate_event(machine=crisis_machine, force_crisis=True))
                else:
                    events.append(generate_event())
            elif GENERATOR_STATE["active_macro"]:
                macro = GENERATOR_STATE["active_macro"]
                affected = [m for m in MACHINES if m["machineId"] in macro["affected_machines"]]
                if affected and random.random() < 0.5:
                    events.append(generate_event(machine=random.choice(affected), force_anomaly=True))
                else:
                    events.append(generate_event())
            else:
                events.append(generate_event())
        
        if crisis_burst_remaining > 0:
            crisis_burst_remaining -= 1
        
        # ── Send ──
        try:
            send_batch(events)
            GENERATOR_STATE["total_sent"] += len(events)
            anomaly_count = sum(1 for e in events if int(e["statusCode"]) >= 1)
            GENERATOR_STATE["anomaly_sent"] += anomaly_count
        except Exception as ex:
            print(f"❌ Send error: {ex}")
        
        time.sleep(INTERVAL_SECONDS)

print("✅ Generator functions loaded.")

In [ ]:
# Cell 5 — Start Background Streaming
if GENERATOR_STATE["running"]:
    print("⚠️  Generator is already running.")
else:
    GENERATOR_STATE["running"] = True
    thread = threading.Thread(target=_generator_loop, daemon=True)
    thread.start()
    print("🚀 Generator running in background.")
    print(f"   Sending {EVENTS_PER_BATCH} events every {INTERVAL_SECONDS}s")
    print(f"   Machines: {', '.join(m['machineName'] for m in MACHINES)}")
    print(f"   Auto-crisis: {AUTO_CRISIS_ENABLED} | Auto-macro: {AUTO_MACRO_ENABLED}")
    print("")
    print("Run the next cell to trigger a crisis on demand.")

In [ ]:
# Cell 6 — Demo Trigger
# ──────────────────────────────────────────────────────────
# Change TRIGGER_TYPE and TRIGGER_TARGET, then run this cell
# to inject a crisis or macro event into the live stream.
# ──────────────────────────────────────────────────────────

TRIGGER_TYPE   = "crisis"     # Options: "crisis" or "macro"
TRIGGER_TARGET = "CNC-007"    # Machine ID for crisis, or macro event ID

# ── Crisis Trigger ──
if TRIGGER_TYPE == "crisis":
    GENERATOR_STATE["crisis_target_machine"] = TRIGGER_TARGET
    GENERATOR_STATE["force_crisis_now"] = True
    target_name = next((m["machineName"] for m in MACHINES if m["machineId"] == TRIGGER_TARGET), TRIGGER_TARGET)
    print(f"🚨 Crisis triggered on: {target_name}")
    print("   Watch the dashboard — anomaly readings will spike within ~3 seconds.")

# ── Macro Event Trigger ──
elif TRIGGER_TYPE == "macro":
    macro = next((m for m in MACRO_EVENTS if m["id"] == TRIGGER_TARGET), None)
    if macro:
        GENERATOR_STATE["active_macro"] = macro
        print(f"🌍 Macro event activated: {macro['label']}")
        print(f"   Affected machines: {macro['affected_machines']}")
        print(f"   Effect: {macro['description']}")
    else:
        print(f"❌ Macro event '{TRIGGER_TARGET}' not found.")
        print(f"   Available: {[m['id'] for m in MACRO_EVENTS]}")

print(f"\n📊 Stats: {GENERATOR_STATE['total_sent']} events sent, {GENERATOR_STATE['anomaly_sent']} anomalies, {GENERATOR_STATE['crisis_triggered']} crises")

In [ ]:
# Cell 7 — Stop Generator (run when done)
GENERATOR_STATE["running"] = False
print("🛑 Generator stopped.")
print(f"📊 Final stats: {GENERATOR_STATE['total_sent']} events sent, {GENERATOR_STATE['anomaly_sent']} anomalies, {GENERATOR_STATE['crisis_triggered']} crises")